# Week 2b.2 — Agent Basics in LangChain

Now that we've covered the API basics in LangChain, we're ready to implement our first agent.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

API key loaded


## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


## 1. The problem

LLMs can serve as the brain but their knowledge and reach are limited.

In [9]:
from pprint import pprint

response = model.invoke("What time is it right now?")
pprint(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'extras': {'signature': 'El4KXAFpFH0TZAp7dCkNoPyWetQPuBW5fZWw3bmUuee5ZYqX7PN8dj96W0ECnj0+86AWdqaGydBs35jmLd3fJHB9wF9gZZowRDf6T+Z113qd55o+QHLhRLIF0RFUNzVW'},
  'text': "I don't know your current local time because I don't have access to "
          'your location or a real-time clock. However, you can easily check '
          "the time on the device you're currently using!",
  'type': 'text'}]


The model has no clock. Its weights were frozen months ago and nothing in the context tells it the time. It can only decline or guess.

We can surpass this limitations by augmenting the LLM with a tool.

## 2. Defining a tool

A tool starts life as an ordinary Python function.

In [6]:
from datetime import datetime

def get_current_time():
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

In [11]:
get_current_time()

'Tuesday, September 22, 2026 at 04:06 PM'

### Requirements for Tool Definition
To make this usable by a model, LangChain needs three things: 
- a name (start with a verb and describe the core of the task)
- a description in the docstring
- an argument schema. 

The `@tool` decorator builds all three from your function. 

**The docstring is not a comment here; the model reads it to decide when to call the tool. Write it carefully.**

In [8]:
# import the tool decorator from langchain's tools module
from langchain.tools import tool

@tool
def get_current_time() -> str:
    """Return the current local date and time.

    Use this whenever the answer depends on today's date or the current time,
    including as a first step before doing any date arithmetic.
    The ISO date (YYYY-MM-DD) is included so it can be passed to other tools.
    """
    now = datetime.now()
    return now.strftime("%A, %B %d, %Y at %I:%M %p") + f" (ISO date: {now.strftime('%Y-%m-%d')})"

Inspect what the decorator built:

In [13]:
print(get_current_time.name)
print(get_current_time.description)
print(get_current_time.args)

get_current_time
Return the current local date and time.

Use this whenever the answer depends on today's date or the current time,
including as a first step before doing any date arithmetic.
The ISO date (YYYY-MM-DD) is included so it can be passed to other tools.
{}


In [14]:
get_current_time.invoke({})

'Tuesday, September 22, 2026 at 04:06 PM (ISO date: 2026-09-22)'

## 3. Give the tool to the model

`bind_tools` attaches tool schemas to the model as a **list** of tools.

The model does not gain the ability to run anything. It only gains the ability to ask.

In [ ]:
# bind the tool to the model (tools are passed as a list)
model_with_tools = model.bind_tools([get_current_time])

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What time is it right now?")
response = model_with_tools.invoke([question])

In [ ]:
pprint(response)

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_61861': 'El4KXAFpFH0TkDaQJR5+6erZ1bGvUmkK4STClzS9Tb3vkSNdIQUyXOYP1rGCQjZT9xKjPwuJzp56DrtenyErU1tdXttsyAjWz9J467ElHfoNlR2w1cYtngZflaRqZ0Es'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0caa8-ab11-7b42-8832-64e768e313b3-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_61861', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 12, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}})


In [ ]:
response.pretty_print()

================================== Ai Message ==================================

[]
Tool Calls:
  get_current_time (call_61861)
 Call ID: call_61861
  Args:


Note: `content` is empty. Instead of answering, the model produced a `tool_calls` entry naming our tool.

Remember the model proposes; it never executes.

In [ ]:
pprint(response.response_metadata)

{'finish_reason': 'STOP',
 'model_name': 'gemini-3.5-flash-lite',
 'model_provider': 'google_genai',
 'safety_ratings': []}


In [ ]:
print(response.tool_calls)

[{'name': 'get_current_time', 'args': {}, 'id': 'call_61861', 'type': 'tool_call'}]



Executing is our job. Run the tool with the call the model requested:

In [ ]:
call = response.tool_calls[0]
tool_result = get_current_time.invoke(call)
print(tool_result)

content='Tuesday, September 22, 2026 at 03:48 PM (ISO date: 2026-09-22)' name='get_current_time' tool_call_id='call_61861'


That produced a `ToolMessage`. Now we can combine the full conversation, including the model's tool call and the tool's result, and send it back:

In [ ]:
messages = [question, response, tool_result]

final = model_with_tools.invoke(messages)


In [ ]:
pprint(final)

AIMessage(content=[{'type': 'text', 'text': 'It is currently 3:48 PM on Tuesday, September 22, 2026.', 'extras': {'signature': 'El4KXAFpFH0TMFeGwvF92Qu4qxfo3IoNO1hnGPkTUtikj8qCDG+myNt+egC1Q77ssScX/UVIsDcHrr+W2HS7l97+8pWHfRMcA0QeCtZBP60b2bqKCd4NyEM/IgLruPXK'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0caa9-919a-7883-a0c8-0feb0ac65bf6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 23, 'total_tokens': 117, 'input_token_details': {'cache_read': 0}})


In [ ]:
print(final.content)

[{'type': 'text', 'text': 'It is currently 3:48 PM on Tuesday, September 22, 2026.', 'extras': {'signature': 'El4KXAFpFH0TMFeGwvF92Qu4qxfo3IoNO1hnGPkTUtikj8qCDG+myNt+egC1Q77ssScX/UVIsDcHrr+W2HS7l97+8pWHfRMcA0QeCtZBP60b2bqKCd4NyEM/IgLruPXK'}}]


### Interim Summary

1. The model saw the question and proposed a tool call.
2. Our code executed the tool.
3. The result went back into the conversation.
4. The model read the result and answered.

Model proposes, runtime executes, result returns. That loop is an agent.

## 4. `create_agent`: Defining the Agentic Loop

The loop so far looked like this:
 propose, execute, return, repeat until the model answers. 
 
 Because this is such a common pattern, LangChain provides a dedicated method to run the loop: `create_agent`
 
 - https://reference.langchain.com/python/langchain/agents/factory/create_agent

 Basic signature: `create_model(model, tools, system_prompt)`


In [9]:
system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""

In [15]:
from langchain.agents import create_agent

#TODO: define an agent with model, tool list, and system prompt
system_prompt = system_prompt

agent = create_agent(model=model,
    tools=[get_current_time],
    system_prompt=system_prompt)


In [18]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What time is it right now?")
result = agent.invoke({"messages": question})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What time is it right now?
================================== Ai Message ==================================

[]
Tool Calls:
  get_current_time (call_72919)
 Call ID: call_72919
  Args:
================================= Tool Message =================================
Name: get_current_time

Tuesday, September 22, 2026 at 04:15 PM (ISO date: 2026-09-22)
================================== Ai Message ==================================

[{'type': 'text', 'text': 'It is currently 4:15 PM on Tuesday, September 22, 2026.', 'extras': {'signature': 'El4KXAFpFH0TAVL8t3Pxjq8LxOFppTnq18GQuBZl7ddCMG6i5iLG6p5WwOMpmyzpW8HXrkz2nGsXvicszWTKOOn/Vm++GjAs/UyFpmuYCVM4bFkwDJ0hiH5xGX3o4A8L'}}]


## 6. Multiple tools

Real agents have several tools and choose between them. Models are bad at date arithmetic, so we'll give our agent a second tool for that.

In [11]:
@tool
def count_days_until_given_date(current_date: str, target_date: str) -> str:
    """Return the number of days from current_date until a future date given as YYYY-MM-DD."""
    target = datetime.strptime(target_date, "%Y-%m-%d").date()
    current = datetime.strptime(current_date, "%Y-%m-%d").date()
    delta = (target - current).days # returns same delta between the two 
    return f"{delta} days"


In [12]:
agent = create_agent(model=model, 
                    tools=[get_current_time,count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )


In [ ]:
question_day = HumanMessage(content="How many days until the midterm on 2026-10-30?")
response = agent.invoke({"messages":question_day })


In [20]:
pprint(response)

Pretty printing has been turned OFF


In [21]:
pprint(response["messages"]) # this is a function list tool demo of how to see the response of the model

Pretty printing has been turned ON


In [22]:
pprint(response["messages"][3].additional_kwargs)

Pretty printing has been turned OFF


The model picked the right tool and filled the argument from the question. Since executing calls is always the same steps, write a small helper:

## 6. Tools that call real APIs

**Motivation for web search:**
GPT-4.1-mini has a knowledge cutoff date of June 1, 2024.
 - https://developers.openai.com/api/docs/models/gpt-4.1-mini

 Qwen3.5's knowledge cutoff date is late 2024 to early 2025.

 Gemini 3.1 Flash-Lite has a knowledge cutoff date of January 2025.

They won't be able to answer questions about events that may have happened after the cutoff date

In [23]:
# same agent as before
agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )

In [25]:
questions = [HumanMessage("Which country won the FIFA World Cup 2026"), 
            HumanMessage("Who won the Super Bowl 2026?"), 
            HumanMessage("Who is the current president of the US")]

In [30]:
result = agent.invoke({"messages": questions[-1]})
print(result)


{'messages': [HumanMessage(content='Who is the current president of the US', additional_kwargs={}, response_metadata={}, id='1a116612-d5c9-4ac4-8f12-1dc1a235038b'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_88212': 'El4KXAFpFH0ToDVQtgxMSis4vDaji2lW2uCGCipRFgft5ftvlscWACnuXOENCcpr8daE8dal4HFNo9D8NIH6WnS+TqawZjZ8+LOYIBXTrdl8paHjrmv5JBDt3QF3FK/X'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0cac3-4976-7210-b7c9-dc90d2f70938-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_88212', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 12, 'total_tokens': 204, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Tuesday, September 22, 2026 at 04:16 PM (ISO date: 2026-09-22)', name='get_c

In [28]:
pprint(result["messages"][-1].content)

Pretty printing has been turned ON


In [ ]:
# loop through questions and then invoke the agent
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    print("")

### Tavily Web Search API

Let's add web search capability to our agent. We'll use Tavily as our web search API.
- https://app.tavily.com/

We'll need to provide an API key in the `.env` file, which has a `TAVILY_API_KEY` field.

#### TavilySearch
Because Tavily is specialized for agentic web search, LangChain supports it.

The `TavilySearch` method queries the Tavily Search API and gets back json.

https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch


In [31]:
from langchain_tavily import TavilySearch
from typing import Dict, Any

@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results


In [32]:
data = search_the_web.invoke({"query":"Who won the super bowl 2026"})

In [33]:
search_docs = data.get("results", data)


In [34]:
pprint(search_docs)

Pretty printing has been turned OFF


Our agent has three tools now:

In [35]:
from langchain.agents import create_agent

agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date, search_the_web],
                    system_prompt = "You're a helpful assistant who answers users' questions concisely. Use the tools available when necessary."
)

In [36]:
result = agent.invoke({"messages": questions[0]})

In [37]:
pprint(result)

Pretty printing has been turned ON


In [39]:
from pprint import pprint

for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    

'Which country won the FIFA World Cup 2026'
[{'extras': {'signature': 'El4KXAFpFH0TP1QKmIODLtxAFpskxB5KXNQ49rsco3fK+XyYLFgSU03AKnhlSr6ypXux9rMD2BnVlOgOmmYXq6fXm7e4MoXGyJ1r+qx7mvvIxc1RjSzo0pSIq3Gp05l+'},
  'text': 'Spain won the 2026 FIFA World Cup, defeating Argentina 1–0 after '
          'extra-time in the final.',
  'type': 'text'}]
'Who won the Super Bowl 2026?'
[{'extras': {'signature': 'El4KXAFpFH0TPRcWYpqANQAJ0Be6AtARM0phEAujozlbc6LbHS+CRWoIPua97jAQx5fqD1Si8jie4ANj6W3r5FS+dtkv3OBprJOoGICxFxVJZzuSWiAUm5BY5d8UATlo'},
  'text': 'The Seattle Seahawks won Super Bowl LX (2026), defeating the New '
          'England Patriots 29–13.',
  'type': 'text'}]
'Who is the current president of the US'
[{'extras': {'signature': 'El4KXAFpFH0TXuCA+swZqD7A+IxrUggFixbqHBCq2kPTfJ+5OZ5jCMC8Muzzu+07Shg+isUsLQmwiTpxbPl25YJLZt7Ro57G4q2y17UHA6T/TtYJ5zczdH1Vzm18GdZx'},
  'text': 'The current president of the United States is Donald Trump (the '
          '47th president).',
  'type': 'text'}]


In [ ]:
# the response from the model
pprint(result["messages"][-1].content)

## 7. ICA: Build a weather tool

Your turn. [wttr.in](https://wttr.in) is a weather service that takes a city name directly and needs no key. One request:

```
https://wttr.in/Boston?format=j1
```

returns JSON. 

The current conditions live in `data["current_condition"][0]`, with fields including `temp_F`, `weatherDesc`, and `windspeedMiles`.

Boston or Boston,MA are acceptable. If city name matches multiple locations, it'll likely default to the one closest to your IP address.

### Your task:
Write a tool `get_weather(city)` that returns the current conditions for that city. Steps:

1. Build the URL from the city argument and fetch it with `requests.get(...).json()`.
2. Pull out temperature, conditions, and wind, and return a readable string.
3. Write the docstring so the model knows when to reach for this tool.
4. Add it to the agent and ask a question that needs it.

Skeleton:

In [40]:
import requests
data = requests.get("https://wttr.in/Boston?format=j1").json()

In [41]:
pprint(data["current_condition"][0])

{'FeelsLikeC': '10',
 'FeelsLikeF': '50',
 'cloudcover': '98',
 'humidity': '59',
 'observation_time': '07:12 PM',
 'precipInches': '0.0',
 'precipMM': '0.0',
 'pressure': '1028',
 'pressureInches': '30',
 'temp_C': '15',
 'temp_F': '60',
 'uvIndex': '3',
 'visibility': '10',
 'visibilityMiles': '6',
 'weatherCode': '122',
 'weatherDesc': [{'value': 'Overcast '}],
 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}],
 'winddir16Point': 'NE',
 'winddirDegree': '52',
 'windspeedKmph': '23',
 'windspeedMiles': '14'}


In [ ]:
import requests

@tool
def get_weather(city: str) -> str:
    """Gets the current weather in a given city using tool calls using agent and displays the current condition of the weather."""
    # TODO: fetch https://wttr.in/<city>?format=j1 with requests.get
    url = f"https://wttr.in/{city}?format=j1"
    
    # TODO: parse the JSON and pull out current_condition and return this segment of the dictionary
    data = requests.get(url).json()

    result = data["current_condition"][0]
    return result

In [ ]:
# Test your tool directly first:
print(get_weather.invoke({"city": "Boston"}))


# # Then give it to the agent:
agent = create_agent(model=model, tools=[get_current_time, count_days_until_given_date, search_the_web, get_weather])
result = agent.invoke({"messages": [HumanMessage(content="Should I bring a jacket in Boston tonight?")]})
print(result["messages"][-1].content)

{'FeelsLikeC': '10', 'FeelsLikeF': '50', 'cloudcover': '98', 'humidity': '59', 'observation_time': '07:12 PM', 'precipInches': '0.0', 'precipMM': '0.0', 'pressure': '1028', 'pressureInches': '30', 'temp_C': '15', 'temp_F': '60', 'uvIndex': '3', 'visibility': '10', 'visibilityMiles': '6', 'weatherCode': '122', 'weatherDesc': [{'value': 'Overcast '}], 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}], 'winddir16Point': 'NE', 'winddirDegree': '52', 'windspeedKmph': '23', 'windspeedMiles': '14'}
[{'type': 'text', 'text': "Yes, you should definitely bring a jacket! It is currently overcast with temperatures around 60°F (15°C), but it feels closer to 50°F (10°C) with some brisk winds coming in from the northeast. As the sun goes down tonight, it's going to get quite chilly.", 'extras': {'signature': 'El4KXAFpFH0TfwWQPiETMT29qQbuASU1RsDxQZbm8KB7BAHQvEJ1liA0PK62G7AXMHzfuwJNNGOWH9pFNuEculXbi9iOuch8j1y6XrRWf1Bmtu6TftEC3c